# ACRS — Layer 3: supervisory agents

Run last (after `ACRS_Evaluation.ipynb`).

Runtime: the 2,000-loan demonstration computes SHAP for every loan.

In [29]:
import os, json, pickle, time, hashlib, csv, platform
from collections import deque
from datetime import datetime
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import roc_auc_score

OUT = 'acrs_data'
SEED = 42
REFER_SHARE = 0.05        # human-review capacity: 5% of applications go to the grey zone
DEMO_N = 2000             # loans in the end-to-end demonstration with per-loan SHAP
STREAM_LEN, CHANGE_AT, N_REPS = 5000, 2500, 100   # detector validation experiments

X_train = pd.read_parquet(f'{OUT}/X_train.parquet')
X_val = pd.read_parquet(f'{OUT}/X_val.parquet')
X_test = pd.read_parquet(f'{OUT}/X_test.parquet')
y_val, y_test = np.load(f'{OUT}/y_val.npy'), np.load(f'{OUT}/y_test.npy')
meta_val, meta_test = pd.read_parquet(f'{OUT}/meta_val.parquet'), pd.read_parquet(f'{OUT}/meta_test.parquet')
models = {}
for key in ['xgb', 'lgbm', 'rf']:
    with open(f'{OUT}/{key}_final.pkl', 'rb') as f:
        models[key] = pickle.load(f)
pd_val = np.load(f'{OUT}/ensemble_prob_val.npy').astype(float)
pd_test = np.load(f'{OUT}/ensemble_prob_test.npy').astype(float)
assert meta_test['issue_d'].is_monotonic_increasing, "Test loans must be in issue-date order — re-run ACRS.ipynb"

# Issue dates are recorded by month only, so the true order of applications within a month is unknown.
# In loan.csv, similar loans sit next to each other within a month; read in that order, the score-based monitors
# would react to the file layout rather than to drift. Applications are therefore placed in random order within
# each issue month (fixed seed). The file order is kept only to report its effect on the monitors (Section 4).
pd_test_file_order = pd_test.copy()
_month_key = meta_test['issue_d'].dt.to_period('M')
_rng = np.random.RandomState(SEED)
order = np.concatenate([_rng.permutation(ix) for _, ix in sorted(meta_test.groupby(_month_key).indices.items())])
X_test = X_test.iloc[order].reset_index(drop=True)
meta_test = meta_test.iloc[order].reset_index(drop=True)
y_test, pd_test = y_test[order], pd_test[order]
assert meta_test['issue_d'].is_monotonic_increasing and len(order) == len(y_test)
pnl_val = (meta_val['total_pymnt'] - meta_val['funded_amnt']).to_numpy()     # realised profit/loss, evaluation only
pnl_test = (meta_test['total_pymnt'] - meta_test['funded_amnt']).to_numpy()
print(f"Validation {len(y_val):,} loans | test {len(y_test):,} loans, {meta_test['issue_d'].min():%b %Y} to {meta_test['issue_d'].max():%b %Y}")

Validation 373,412 loans | test 480,812 loans, Jan 2016 to Dec 2018


## 1. Decision thresholds, chosen on the 2015 validation loans
Realised profit/loss of approving a loan = total paid by the borrower − amount funded (used only for evaluation).
`t_star` maximises the validation profit of "approve if PD < t". The grey zone around it holds `REFER_SHARE` of applications.

In [31]:
grid = np.unique(np.quantile(pd_val, np.linspace(0.01, 0.99, 197)))
profit = np.array([pnl_val[pd_val < t].sum() for t in grid])
t_star = float(grid[profit.argmax()])
q_star = float((pd_val < t_star).mean())
t_low = float(np.quantile(pd_val, max(q_star - REFER_SHARE / 2, 0)))
t_high = float(np.quantile(pd_val, min(q_star + REFER_SHARE / 2, 1)))
policy = {'t_star': round(t_star, 4), 't_low': round(t_low, 4), 't_high': round(t_high, 4), 'refer_share_target': REFER_SHARE,
          'chosen_on': 'validation (2015), realised profit/loss',
          'validation_profit_approve_all': round(float(pnl_val.sum()), 0),
          'validation_profit_approve_below_t_star': round(float(profit.max()), 0)}
print(json.dumps(policy, indent=2))
print(f"Rule: APPROVE if PD < {t_low:.3f}; REFER_TO_HUMAN if {t_low:.3f} <= PD < {t_high:.3f} or a real-time drift alarm; DECLINE if PD >= {t_high:.3f}")

def band(p):
    return np.where(p < t_low, 'Low', np.where(p < t_high, 'Medium', 'High'))

{
  "t_star": 0.2776,
  "t_low": 0.2621,
  "t_high": 0.2955,
  "refer_share_target": 0.05,
  "chosen_on": "validation (2015), realised profit/loss",
  "validation_profit_approve_all": 191249698.0,
  "validation_profit_approve_below_t_star": 274347415.0
}
Rule: APPROVE if PD < 0.262; REFER_TO_HUMAN if 0.262 <= PD < 0.295 or a real-time drift alarm; DECLINE if PD >= 0.295


## 2. Agent 2 — real-time, label-free monitors (scores only) and the delayed-label DDM
* `WindowMean`: alarm when the mean PD of the last 500 applications moves more than `threshold` from the reference
  (mean validation PD); after an alarm the reference is reset to the current window. Fixed window, inspired by ADWIN.
* `PageHinkley`: alarm on a sustained **upward** shift of the PD stream.
* `DDM` (Gama et al., 2004): monitors the error rate of decisions once outcomes are known. Standard version: warning at
  2 standard deviations, drift at 3, reset after drift. It needs realised defaults, so it is **delayed-label** monitoring.

In [33]:
class WindowMean:
    def __init__(self, reference, window=500, threshold=0.03):
        self.ref, self.window, self.threshold = reference, window, threshold
        self.buf, self.total = deque(), 0.0
    def update(self, x):
        self.buf.append(x); self.total += x
        if len(self.buf) > self.window:
            self.total -= self.buf.popleft()
        if len(self.buf) == self.window:
            m = self.total / self.window
            if abs(m - self.ref) > self.threshold:
                self.ref = m
                return True
        return False

class PageHinkley:
    def __init__(self, delta=0.005, lam=1.0):
        self.delta, self.lam = delta, lam
        self.reset()
    def reset(self):
        self.n, self.mean, self.cum, self.min_cum = 0, 0.0, 0.0, 0.0
    def update(self, x):
        self.n += 1
        self.mean += (x - self.mean) / self.n
        self.cum += x - self.mean - self.delta
        self.min_cum = min(self.min_cum, self.cum)
        if self.cum - self.min_cum > self.lam:
            self.reset()
            return True
        return False

class DDM:
    def __init__(self, warn=2.0, drift=3.0, min_n=30):
        self.warn, self.drift, self.min_n = warn, drift, min_n
        self.reset()
    def reset(self):
        self.n, self.errors, self.p_min, self.s_min = 0, 0, np.inf, np.inf
    def update(self, error):
        self.n += 1; self.errors += error
        if self.n < self.min_n:
            return 'OK'
        p = self.errors / self.n
        s = np.sqrt(p * (1 - p) / self.n)
        if p + s < self.p_min + self.s_min:
            self.p_min, self.s_min = p, s
        if p + s > self.p_min + self.drift * self.s_min:
            self.reset()
            return 'DRIFT'
        return 'WARNING' if p + s > self.p_min + self.warn * self.s_min else 'OK'

def run(detector_factory, scores, errors=None):
    """Indices where a detector raises an alarm on one stream."""
    d = detector_factory()
    if errors is None:
        return [i for i, x in enumerate(scores) if d.update(x)]
    return [i for i, e in enumerate(errors) if d.update(int(e)) == 'DRIFT']

# Calibrate the label-free thresholds on no-change streams from the first half of 2015 (99th percentile of the
# largest statistic seen in a 5,000-loan stream), so a no-change stream raises a false alarm about 1% of the time.
rng = np.random.RandomState(SEED)
h1 = np.where(meta_val['issue_d'].dt.month <= 6)[0]
h2 = np.where(meta_val['issue_d'].dt.month > 6)[0]
ref_mean = float(pd_val.mean())
max_dev, max_ph = [], []
for _ in range(N_REPS):
    s = pd_val[rng.choice(h1, STREAM_LEN, replace=len(h1) < STREAM_LEN)]
    roll = np.convolve(s, np.ones(500) / 500, mode='valid')
    max_dev.append(np.abs(roll - ref_mean).max())
    ph = PageHinkley(delta=0.005, lam=np.inf); best = 0.0
    for x in s:
        ph.update(x); best = max(best, ph.cum - ph.min_cum)
    max_ph.append(best)
WINDOW_THRESHOLD = float(np.quantile(max_dev, 0.99))
PH_LAMBDA = float(np.quantile(max_ph, 0.99))
monitor_settings = {'window': 500, 'window_threshold': round(WINDOW_THRESHOLD, 4), 'window_reference_mean_pd': round(ref_mean, 4),
                    'ph_delta': 0.005, 'ph_lambda': round(PH_LAMBDA, 3), 'ddm_warning_sd': 2.0, 'ddm_drift_sd': 3.0,
                    'calibrated_on': 'no-change streams from Jan–Jun 2015 (99th percentile of the maximum statistic)'}
print(json.dumps(monitor_settings, indent=2))
make_window = lambda: WindowMean(ref_mean, 500, WINDOW_THRESHOLD)
make_ph = lambda: PageHinkley(0.005, PH_LAMBDA)
make_ddm = lambda: DDM(2.0, 3.0)

{
  "window": 500,
  "window_threshold": 0.026,
  "window_reference_mean_pd": 0.1723,
  "ph_delta": 0.005,
  "ph_lambda": 10.48,
  "ddm_warning_sd": 2.0,
  "ddm_drift_sd": 3.0,
  "calibrated_on": "no-change streams from Jan\u2013Jun 2015 (99th percentile of the maximum statistic)"
}


## 3. Detector validation on streams with a known answer
* **No change**: random streams of Jul–Dec 2015 loans (not used for calibration) → every alarm is a false alarm.
* **Real change**: 2,500 loans from 2015, then 2,500 loans from 2016–2018 → change point known (loan 2,500).
* **Injected change**: 2,500 loans from Jul–Dec 2015, then 2,500 loans from the same months but grades D–G only (riskier borrower mix).
Detection delay = loans between the change point and the first alarm after it. 100 random streams each.

In [35]:
err_val = ((pd_val >= t_star).astype(int) != y_val).astype(int)
err_test = ((pd_test >= t_star).astype(int) != y_test).astype(int)
risky_h2 = np.intersect1d(h2, np.where(meta_val['grade'].isin(list('DEFG')))[0])

def make_stream(name, r):
    if name == 'no_change':
        i = r.choice(h2, STREAM_LEN)
        return pd_val[i], err_val[i]
    if name == 'real_change_2015_to_2016_18':
        a, b = r.choice(len(y_val), CHANGE_AT), r.choice(len(y_test), STREAM_LEN - CHANGE_AT)
        return np.r_[pd_val[a], pd_test[b]], np.r_[err_val[a], err_test[b]]
    a, b = r.choice(h2, CHANGE_AT), r.choice(risky_h2, STREAM_LEN - CHANGE_AT)   # injected_riskier_mix (same period)
    return np.r_[pd_val[a], pd_val[b]], np.r_[err_val[a], err_val[b]]

validation = []
for name in ['no_change', 'real_change_2015_to_2016_18', 'injected_riskier_mix']:
    r = np.random.RandomState(SEED + 1)
    stats = {k: {'false_alarms': 0, 'detected': 0, 'delays': []} for k in ['window', 'page_hinkley', 'ddm_delayed_labels']}
    for _ in range(N_REPS):
        s, e = make_stream(name, r)
        for k, alarms in [('window', run(make_window, s)), ('page_hinkley', run(make_ph, s)),
                          ('ddm_delayed_labels', run(make_ddm, None, e))]:
            if name == 'no_change':
                stats[k]['false_alarms'] += len(alarms)
            else:
                stats[k]['false_alarms'] += sum(i < CHANGE_AT for i in alarms)
                after = [i for i in alarms if i >= CHANGE_AT]
                if after:
                    stats[k]['detected'] += 1
                    stats[k]['delays'].append(after[0] - CHANGE_AT)
    pre_len = STREAM_LEN if name == 'no_change' else CHANGE_AT
    for k, v in stats.items():
        validation.append({'scenario': name, 'detector': k,
                           'false_alarms_per_10k_loans': round(v['false_alarms'] / (N_REPS * pre_len) * 1e4, 2),
                           'detection_rate': None if name == 'no_change' else round(v['detected'] / N_REPS, 2),
                           'median_delay_loans': None if not v['delays'] else int(np.median(v['delays']))})
validation_df = pd.DataFrame(validation)
validation_df.to_csv(f'{OUT}/drift_detector_validation.csv', index=False)
print(validation_df.to_string(index=False))

                   scenario           detector  false_alarms_per_10k_loans  detection_rate  median_delay_loans
                  no_change             window                        0.00             NaN                 NaN
                  no_change       page_hinkley                        0.00             NaN                 NaN
                  no_change ddm_delayed_labels                        0.68             NaN                 NaN
real_change_2015_to_2016_18             window                        0.00            0.00                 NaN
real_change_2015_to_2016_18       page_hinkley                        0.00            0.00                 NaN
real_change_2015_to_2016_18 ddm_delayed_labels                        1.16            0.18              1828.0
       injected_riskier_mix             window                        0.00            1.00               104.0
       injected_riskier_mix       page_hinkley                        0.00            1.00                72.0
 

## 4. The chronological test stream (Jan 2016 – Dec 2018)
Real-time monitors run on the PD stream as applications arrive (random order within each issue month, see Section 0).
DDM runs offline on the same stream once outcomes are known. For transparency, the monitors are also run on the
raw file order, to show how much the order of arrival alone changes the number of alarms.

In [37]:
rt_window = set(run(make_window, pd_test))
rt_ph = set(run(make_ph, pd_test))
ddm_alarms = run(make_ddm, None, err_test)
rt_alarm = np.zeros(len(pd_test), bool)
rt_alarm[list(rt_window | rt_ph)] = True
month = meta_test['issue_d'].dt.to_period('M').astype(str).to_numpy()
stream_summary = {'order_within_month': 'random (fixed seed); issue dates are monthly',
                  'window_alarms': len(rt_window), 'page_hinkley_alarms': len(rt_ph), 'ddm_delayed_label_alarms': len(ddm_alarms),
                  'file_order_window_alarms': len(run(make_window, pd_test_file_order)),
                  'file_order_page_hinkley_alarms': len(run(make_ph, pd_test_file_order)),
                  'window_alarm_months': sorted(set(month[list(rt_window)])), 'ddm_alarm_months': sorted(set(month[ddm_alarms]))}
print(json.dumps(stream_summary, indent=2))

monthly = pd.DataFrame({'month': month, 'y': y_test, 'pd': pd_test}).groupby('month').apply(
    lambda g: pd.Series({'loans': len(g), 'default_rate': g['y'].mean(), 'mean_pd': g['pd'].mean(),
                         'roc_auc': roc_auc_score(g['y'], g['pd']) if g['y'].nunique() == 2 else np.nan}), include_groups=False)
monthly.round(4).to_csv(f'{OUT}/delayed_label_monitoring_by_month.csv')
print("Delayed-label monitoring by issue month (first rows):")
print(monthly.head(6).round(3).to_string())

{
  "order_within_month": "random (fixed seed); issue dates are monthly",
  "window_alarms": 3,
  "page_hinkley_alarms": 3,
  "ddm_delayed_label_alarms": 1,
  "file_order_window_alarms": 226,
  "file_order_page_hinkley_alarms": 48,
  "window_alarm_months": [
    "2016-01",
    "2018-12"
  ],
  "ddm_alarm_months": [
    "2016-03"
  ]
}
Delayed-label monitoring by issue month (first rows):
           loans  default_rate  mean_pd  roc_auc
month                                           
2016-01  27985.0         0.193    0.159    0.755
2016-02  32076.0         0.207    0.163    0.740
2016-03  40701.0         0.256    0.175    0.721
2016-04  22997.0         0.263    0.169    0.716
2016-05  17394.0         0.268    0.169    0.714
2016-06  19978.0         0.247    0.163    0.711


## 5. Agent 3 decisions on the whole test set, with errors, sensitivity and profit/loss

In [39]:
def prf(flag, labels):
    """Precision, recall and F1 when `flag` marks the loans treated as predicted defaults."""
    pos = labels == 1
    tp, fp, fn = int((flag & pos).sum()), int((flag & ~pos).sum()), int((~flag & pos).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    f1 = 2 * prec * rec / (prec + rec) if tp else np.nan
    return prec, rec, f1

def evaluate_policy(p, labels, pnl, lo, hi, alarm=None):
    refer = (p >= lo) & (p < hi)
    if alarm is not None:
        refer |= alarm
    approve = (p < lo) & ~refer
    decline = (p >= hi) & ~refer
    n = len(p)
    out = {'approve_rate': approve.mean(), 'refer_rate': refer.mean(), 'decline_rate': decline.mean(),
           'default_rate_approved': labels[approve].mean() if approve.any() else np.nan,
           'default_rate_referred': labels[refer].mean() if refer.any() else np.nan,
           'default_rate_declined': labels[decline].mean() if decline.any() else np.nan,
           'approval_errors': int((approve & (labels == 1)).sum()),
           'decline_errors': int((decline & (labels == 0)).sum()),
           'profit_if_referred_declined': pnl[approve].sum(),
           'profit_if_referred_approved': pnl[approve | refer].sum()}
    out['share_of_defaults_caught'] = 1 - out['approval_errors'] / max(int((labels == 1).sum()), 1)
    # F1 at the decision cut-offs: 'decline' = declined loans are the predicted defaults;
    # 'flagged' = declined or referred loans are the predicted defaults
    out['precision_decline'], out['recall_decline'], out['f1_decline'] = prf(decline, labels)
    out['precision_flagged'], out['recall_flagged'], out['f1_flagged'] = prf(decline | refer, labels)
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in out.items()}

decisions_full = evaluate_policy(pd_test, y_test, pnl_test, t_low, t_high, rt_alarm)
baselines = {'approve_all_profit': round(float(pnl_test.sum()), 0),
             'single_threshold_t_star_profit': round(float(pnl_test[pd_test < t_star].sum()), 0),
             'single_threshold_t_star_f1': round(float(prf(pd_test >= t_star, y_test)[2]), 4)}
print("Policy on all test loans:")
print(json.dumps(decisions_full, indent=2))
print("Baselines:", baselines)

sens = []
for share in [0.02, 0.05, 0.10]:
    for shift in [-0.10, -0.05, 0.0, 0.05, 0.10]:
        q = min(max(q_star + shift, share / 2), 1 - share / 2)
        lo, hi = np.quantile(pd_val, q - share / 2), np.quantile(pd_val, q + share / 2)
        r = evaluate_policy(pd_test, y_test, pnl_test, lo, hi)
        sens.append({'refer_share_target': share, 'quantile_shift': shift, 't_low': round(float(lo), 4), 't_high': round(float(hi), 4),
                     **{k: r[k] for k in ['approve_rate', 'refer_rate', 'decline_rate', 'default_rate_approved',
                                          'approval_errors', 'decline_errors', 'profit_if_referred_declined']}})
sens_df = pd.DataFrame(sens)
sens_df.to_csv(f'{OUT}/policy_threshold_sensitivity.csv', index=False)
print("\nThreshold sensitivity (no drift referrals):")
print(sens_df.round(4).to_string(index=False))

by_year_dec = []
for yr in sorted(meta_test['issue_d'].dt.year.unique()):
    m = (meta_test['issue_d'].dt.year == yr).to_numpy()
    by_year_dec.append({'year': int(yr), **evaluate_policy(pd_test[m], y_test[m], pnl_test[m], t_low, t_high, rt_alarm[m])})
pd.DataFrame(by_year_dec).to_csv(f'{OUT}/policy_by_year.csv', index=False)

Policy on all test loans:
{
  "approve_rate": 0.8237,
  "refer_rate": 0.0504,
  "decline_rate": 0.126,
  "default_rate_approved": 0.1828,
  "default_rate_referred": 0.3724,
  "default_rate_declined": 0.4725,
  "approval_errors": 72386,
  "decline_errors": 31950,
  "profit_if_referred_declined": -128518501.6741,
  "profit_if_referred_approved": -180596869.7033,
  "share_of_defaults_caught": 0.3421,
  "precision_decline": 0.4725,
  "recall_decline": 0.2601,
  "f1_decline": 0.3355,
  "precision_flagged": 0.4439,
  "recall_flagged": 0.3421,
  "f1_flagged": 0.3864
}
Baselines: {'approve_all_profit': -401946965.0, 'single_threshold_t_star_profit': -153202891.0, 'single_threshold_t_star_f1': 0.3633}

Threshold sensitivity (no drift referrals):
 refer_share_target  quantile_shift  t_low  t_high  approve_rate  refer_rate  decline_rate  default_rate_approved  approval_errors  decline_errors  profit_if_referred_declined
               0.02           -0.10 0.2204  0.2290        0.7342      0.0217 

## 6. The three agents, end to end, on the first 2,000 test loans (with per-loan SHAP, hash-chained log and timings)

In [41]:
background = X_train.sample(n=min(100, len(X_train)), random_state=SEED)
models['xgb'].set_params(enable_categorical=False)
explainers = {
    'xgb': shap.TreeExplainer(models['xgb'], data=background, feature_perturbation='interventional', model_output='probability'),
    'lgbm': shap.TreeExplainer(models['lgbm'], data=background, feature_perturbation='interventional', model_output='probability'),
    'rf': shap.TreeExplainer(models['rf'], data=background, feature_perturbation='interventional', model_output='raw'),
}
def class1(values):
    if isinstance(values, list):
        return np.asarray(values[1], float)
    values = np.asarray(values, float)
    return values[:, :, 1] if values.ndim == 3 else values

class ScoringAgent:
    """Agent 1: ensemble PD, risk band and the three largest SHAP contributions (probability scale)."""
    def score(self, row):
        pd_ = float(np.mean([m.predict_proba(row)[:, 1][0] for m in models.values()]))
        return pd_, str(band(np.array([pd_]))[0])
    def explain(self, row):
        phi = np.mean([class1(e.shap_values(row, check_additivity=False))[0] for e in explainers.values()], axis=0)
        top = np.argsort(-np.abs(phi))[:3]
        return [(row.columns[j], float(phi[j])) for j in top]

class MonitoringAgent:
    """Agent 2 (real-time part): sees only the PD of each new application, never its outcome."""
    def __init__(self):
        self.window, self.ph = make_window(), make_ph()
    def update(self, pd_):
        w, p = self.window.update(pd_), self.ph.update(pd_)
        return 'WINDOW_ALARM' if w else ('PH_ALARM' if p else 'OK')

class ComplianceAgent:
    """Agent 3: decision rule, human-in-the-loop referral and an append-only, hash-chained audit log."""
    FIELDS = ['timestamp', 'run_id', 'loan_idx', 'issue_month', 'pd', 'risk_band', 'monitor_status', 'hitl_required',
              'hitl_reason', 'top1_feature', 'top1_shap', 'decision', 'prev_hash', 'row_hash']
    def __init__(self, path, run_id):
        self.path, self.run_id, self.prev = path, run_id, '0' * 64
        with open(self.path, 'x', newline='') as f:            # 'x' = never overwrite an existing log
            csv.writer(f).writerow(self.FIELDS)
    def decide(self, idx, issue_month, pd_, band_, status_, top3):
        if band_ == 'Medium':
            decision, hitl, reason = 'REFER_TO_HUMAN', True, 'Grey zone PD'
        elif status_ != 'OK':
            decision, hitl, reason = 'REFER_TO_HUMAN', True, f'Real-time drift alarm ({status_})'
        else:
            decision, hitl, reason = ('AUTO_APPROVE' if band_ == 'Low' else 'AUTO_DECLINE'), False, ''
        row = [datetime.now().isoformat(timespec='milliseconds'), self.run_id, idx, issue_month, round(pd_, 4), band_,
               status_, hitl, reason, top3[0][0], round(top3[0][1], 4), decision, self.prev]
        row_hash = hashlib.sha256('|'.join(map(str, row)).encode()).hexdigest()
        with open(self.path, 'a', newline='') as f:
            csv.writer(f).writerow(row + [row_hash])
        self.prev = row_hash
        return decision, reason

def verify_chain(path):
    prev = '0' * 64
    with open(path, newline='') as f:
        for row in list(csv.reader(f))[1:]:
            if row[-2] != prev or hashlib.sha256('|'.join(row[:-1]).encode()).hexdigest() != row[-1]:
                return False
            prev = row[-1]
    return True

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
audit_path = f'{OUT}/audit_trail_{run_id}.csv'
agent1, agent2, agent3 = ScoringAgent(), MonitoringAgent(), ComplianceAgent(audit_path, run_id)
timings = {'score': [], 'shap': [], 'monitor': [], 'decide_and_log': []}
records = []
t_start = time.time()
for i in range(min(DEMO_N, len(X_test))):
    row = X_test.iloc[[i]]
    t0 = time.perf_counter(); pd_, band_ = agent1.score(row)
    t1 = time.perf_counter(); top3 = agent1.explain(row)
    t2 = time.perf_counter(); status_ = agent2.update(pd_)
    t3 = time.perf_counter(); decision, reason = agent3.decide(int(order[i]), month[i], pd_, band_, status_, top3)
    t4 = time.perf_counter()
    for k, a, b in [('score', t0, t1), ('shap', t1, t2), ('monitor', t2, t3), ('decide_and_log', t3, t4)]:
        timings[k].append(b - a)
    records.append({'loan_idx': int(order[i]), 'issue_month': month[i], 'true_label': 'DEFAULT' if y_test[i] else 'PAID', 'pd': round(pd_, 4),
                    'band': band_, 'monitor': status_, 'decision': decision, 'reason': reason,
                    'top3': [(f, round(v, 4)) for f, v in top3]})
    if (i + 1) % 250 == 0:
        print(f"  {i + 1:,} loans processed ({time.time() - t_start:.0f}s)")
demo = pd.DataFrame(records)
demo_counts = demo['decision'].value_counts().to_dict()
print(f"\nDemo decisions ({len(demo):,} loans, {demo['issue_month'].iloc[0]} to {demo['issue_month'].iloc[-1]}): {demo_counts}")
print(f"Audit log: {audit_path} | hash chain valid: {verify_chain(audit_path)}")

  250 loans processed (80s)
  500 loans processed (158s)
  750 loans processed (235s)
  1,000 loans processed (314s)
  1,250 loans processed (393s)
  1,500 loans processed (471s)
  1,750 loans processed (549s)
  2,000 loans processed (626s)

Demo decisions (2,000 loans, 2016-01 to 2016-01): {'AUTO_APPROVE': 1684, 'AUTO_DECLINE': 228, 'REFER_TO_HUMAN': 88}
Audit log: acrs_data/audit_trail_20261121_193025.csv | hash chain valid: True


## 7. Latency benchmark

In [43]:
lat = {}
for k, v in timings.items():
    v = np.array(v) * 1000
    lat[k] = {'mean_ms': round(float(v.mean()), 2), 'median_ms': round(float(np.median(v)), 2), 'p95_ms': round(float(np.quantile(v, 0.95)), 2)}
total = np.sum([np.array(v) for v in timings.values()], axis=0) * 1000
lat['layer3_total_per_loan'] = {'mean_ms': round(float(total.mean()), 2), 'median_ms': round(float(np.median(total)), 2),
                                'p95_ms': round(float(np.quantile(total, 0.95)), 2)}
n_batch = min(100_000, len(X_test))
t0 = time.perf_counter()
_ = np.mean([m.predict_proba(X_test.iloc[:n_batch])[:, 1] for m in models.values()], axis=0)
batch_s = time.perf_counter() - t0
lat['layer1_batch_scoring'] = {'loans': n_batch, 'seconds': round(batch_s, 2), 'loans_per_second': int(n_batch / batch_s)}
try:
    import psutil
    lat['process_memory_mb'] = round(psutil.Process().memory_info().rss / 2**20, 0)
except ImportError:
    lat['process_memory_mb'] = None
lat['hardware'] = {'platform': platform.platform(), 'processor': platform.processor(), 'cpu_count': os.cpu_count()}
print(json.dumps(lat, indent=2))

{
  "score": {
    "mean_ms": 77.84,
    "median_ms": 75.44,
    "p95_ms": 88.1
  },
  "shap": {
    "mean_ms": 233.63,
    "median_ms": 227.02,
    "p95_ms": 297.63
  },
  "monitor": {
    "mean_ms": 0.01,
    "median_ms": 0.01,
    "p95_ms": 0.02
  },
  "decide_and_log": {
    "mean_ms": 1.21,
    "median_ms": 0.77,
    "p95_ms": 1.06
  },
  "layer3_total_per_loan": {
    "mean_ms": 312.69,
    "median_ms": 305.57,
    "p95_ms": 379.86
  },
  "layer1_batch_scoring": {
    "loans": 100000,
    "seconds": 1.22,
    "loans_per_second": 81763
  },
  "process_memory_mb": 1272.0,
  "hardware": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
    "cpu_count": 12
  }
}


## 8. Tables for Chapter 4 and the results file

In [45]:
print("Table 4.8 — Agent 1 output for the first three demo loans")
for _, r in demo.head(3).iterrows():
    print(f"  #{r['loan_idx']} {r['issue_month']} {r['true_label']:<7} PD {r['pd']:.4f} {r['band']:<6} -> {r['decision']:<15} top-3: {r['top3']}")
print("\nTable 4.10 — audit log extract (first referral and its neighbours)")
audit = pd.read_csv(audit_path)
ref_rows = audit.index[audit['decision'] == 'REFER_TO_HUMAN']
k = int(ref_rows[0]) if len(ref_rows) else 1
print(audit.iloc[max(k - 1, 0):k + 2][['timestamp', 'loan_idx', 'pd', 'risk_band', 'monitor_status', 'hitl_required',
                                        'hitl_reason', 'top1_feature', 'top1_shap', 'decision']].to_string(index=False))

l2 = json.load(open(f'{OUT}/layer2_results.json'))
ev = json.load(open(f'{OUT}/evaluation_results.json'))
ens_test = next(r for r in ev['discrimination']['test'] if r['model'] == 'Ensemble (L1)')
comparison = {
    'ROC-AUC (test)': [ens_test['roc_auc']] * 3, 'KS (test)': [ens_test['ks']] * 3, 'Brier (test)': [ens_test['brier']] * 3,
    'PSI monitoring (train vs test)': ['—'] + [next(r['psi_train_vs_test'] for r in l2['psi'] if r['model'] == 'Ensemble')] * 2,
    'Batch SHAP (probability scale)': ['—', '2,000 loans', '2,000 loans'],
    'Real-time drift monitoring (label-free)': ['—', '—', f"window + Page-Hinkley: {len(rt_window) + len(rt_ph)} alarms on {len(y_test):,} loans"],
    'Delayed-label monitoring': ['—', '—', f"DDM: {len(ddm_alarms)} alarms; monthly ROC-AUC/calibration"],
    'Per-loan explanation': ['—', '—', 'top-3 SHAP per decision'],
    'Human referral': ['—', '—', f"{decisions_full['refer_rate']:.1%} of test loans"],
    'Audit log': ['—', '—', 'append-only, hash-chained'],
}
print("\nTable 4.11 — three-layer comparison (predictive metrics identical by construction: same ensemble in every layer)")
print(pd.DataFrame(comparison, index=['Layer 1', 'Layer 2', 'Layer 3']).T.to_string())

l3 = {'layer': 3, 'policy': policy, 'monitor_settings': monitor_settings, 'detector_validation': validation,
      'test_stream': stream_summary, 'decisions_full_test': decisions_full, 'baselines': baselines,
      'demo': {'loans': int(len(demo)), 'decisions': demo_counts, 'first_month': demo['issue_month'].iloc[0],
               'last_month': demo['issue_month'].iloc[-1], 'audit_file': os.path.basename(audit_path),
               'hash_chain_valid': verify_chain(audit_path), 'first_three': demo.head(3).to_dict('records')},
      'latency': lat}
with open(f'{OUT}/layer3_results.json', 'w') as f:
    json.dump(l3, f, indent=2, default=str)
print("\nSaved: layer3_results.json, drift_detector_validation.csv, policy_threshold_sensitivity.csv, policy_by_year.csv,")
print(f"       delayed_label_monitoring_by_month.csv, {os.path.basename(audit_path)}")
print("Layer 3 COMPLETE — all thesis numbers now come from acrs_data/*.json and *.csv")

Table 4.8 — Agent 1 output for the first three demo loans
  #23138 2016-01 DEFAULT PD 0.2291 Low    -> AUTO_APPROVE    top-3: [('acc_open_past_24mths', 0.0403), ('sub_grade', 0.0292), ('term', -0.0203)]
  #20615 2016-01 PAID    PD 0.0429 Low    -> AUTO_APPROVE    top-3: [('sub_grade', -0.034), ('int_rate', -0.0171), ('grade', -0.0139)]
  #27154 2016-01 PAID    PD 0.4151 High   -> AUTO_DECLINE    top-3: [('sub_grade', 0.077), ('term', 0.065), ('grade', 0.0335)]

Table 4.10 — audit log extract (first referral and its neighbours)
              timestamp  loan_idx     pd risk_band monitor_status  hitl_required  hitl_reason top1_feature  top1_shap       decision
2026-11-21T19:30:34.450      1103 0.0918       Low             OK          False          NaN    sub_grade    -0.0202   AUTO_APPROVE
2026-11-21T19:30:34.771      7978 0.2677    Medium             OK           True Grey zone PD    sub_grade     0.0452 REFER_TO_HUMAN
2026-11-21T19:30:35.050     11331 0.0849       Low             OK   